# 📊 Notebook 06 — EV Power BI Dashboard Preparation

## Overview

This notebook converts the cleaned and engineered EV dataset into **Power BI-ready analytical tables**.

### Objective

Build a professional dashboard data layer containing:

- Executive KPIs
- EV market overview
- Manufacturer analysis
- Model analysis
- State/geographic analysis
- EV type analysis
- Model-year trends
- Electric-range analysis
- MSRP analysis
- CAFV eligibility
- Vehicle-age analysis
- Slicer-ready dimensions
- Power BI export files

### Dashboard design philosophy

The final Power BI report should support:

**Overview → Market → Manufacturers → Models → Geography → Technology → Vehicle Economics**

The dashboard should use a **cream/light theme**, strong KPI cards, clean charts, interactive slicers, drill-through where useful, and consistent business terminology.

> This notebook does not scrape new data and does not redo the cleaning performed in Notebook 01.


## 1. Input and Output

**Input**

```text
../datas/cleaned/ev_cleaned.csv
```

**Output**

```text
../datas/powerbi/
├── ev_powerbi_fact.csv
├── dim_date.csv
├── dim_make.csv
├── dim_model.csv
├── dim_state.csv
├── dim_ev_type.csv
├── dim_cafv.csv
├── manufacturer_summary.csv
├── model_summary.csv
├── state_summary.csv
├── yearly_summary.csv
├── ev_type_summary.csv
└── powerbi_kpis.csv
```

These files are designed for import into Power BI.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("../datas/cleaned/ev_cleaned.csv")
OUTPUT_DIR = Path("../datas/powerbi")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("Dataset loaded:", df.shape)
display(df.head())


Dataset loaded: (150482, 17)


C:\Users\Admin\AppData\Local\Temp\ipykernel_20128\1341173025.py:12: DtypeWarning: Columns (0: Postal_Code, 1: Census_Tract) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


,VIN,County,City,State,Postal_Code,Model_Year,Make,Model,EV_Type,CAFV_Eligibility,Electric_Range,Base_MSRP,Vehicle_ID,Vehicle_Location,Electric_Utility,Census_Tract,Legislative_District
0,KM8K33AGXL,King,Seattle,WA,"98,103.00",2020,HYUNDAI,KONA,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,258,0,249675142,POINT (-122.34301 47.659185),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),"53,033,004,800.00",43
1,1C4RJYB61N,King,Bothell,WA,"98,011.00",2022,JEEP,GRAND CHEROKEE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,25,0,233928502,POINT (-122.20578 47.762405),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),"53,033,021,804.00",1
2,1C4RJYD61P,Yakima,Yakima,WA,"98,908.00",2023,JEEP,GRAND CHEROKEE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,25,0,229675939,POINT (-120.6027202 46.5965625),PACIFICORP,"53,077,002,900.00",14
3,5YJ3E1EA7J,King,Kirkland,WA,"98,034.00",2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215,0,104714466,POINT (-122.209285 47.71124),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),"53,033,021,903.00",45
4,WBY7Z8C5XJ,Thurston,Olympia,WA,"98,501.00",2018,BMW,I3,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,97,0,185498386,POINT (-122.89692 47.043535),PUGET SOUND ENERGY INC,"53,067,010,700.00",22


## 2. Validate the Cleaned Dataset

Only validation is performed here. Cleaning belongs to Notebook 01.


In [2]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:")
display(df.dtypes)


Rows: 150482
Columns: 17

Missing values:


VIN                     0
County                  0
City                    0
State                   0
Postal_Code             0
Model_Year              0
Make                    0
Model                   0
EV_Type                 0
CAFV_Eligibility        0
Electric_Range          0
Base_MSRP               0
Vehicle_ID              0
Vehicle_Location        0
Electric_Utility        0
Census_Tract            0
Legislative_District    0
dtype: int64


Duplicate rows: 0

Data types:


VIN                        str
County                     str
City                       str
State                      str
Postal_Code             object
Model_Year               int64
Make                       str
Model                      str
EV_Type                    str
CAFV_Eligibility           str
Electric_Range           int64
Base_MSRP                int64
Vehicle_ID               int64
Vehicle_Location           str
Electric_Utility           str
Census_Tract            object
Legislative_District     int64
dtype: object

## 3. Create Dashboard Features

These are analytical features required by the dashboard.

### Main derived fields

- Vehicle_Age
- Age_Group
- Range_Category
- MSRP_Category
- CAFV_Flag
- Model_Year_Group

These features make Power BI slicers and visualizations easier to build.


In [3]:
REFERENCE_YEAR = int(df["Model_Year"].max())

df["Vehicle_Age"] = (REFERENCE_YEAR - df["Model_Year"]).clip(lower=0)

df["Age_Group"] = pd.cut(
    df["Vehicle_Age"],
    bins=[-1, 2, 5, 10, np.inf],
    labels=["0–2 Years", "3–5 Years", "6–10 Years", "10+ Years"]
)

df["Range_Category"] = pd.cut(
    df["Electric_Range"],
    bins=[-1, 100, 200, 300, 400, np.inf],
    labels=["0–100", "101–200", "201–300", "301–400", "400+"]
)

df["MSRP_Category"] = pd.cut(
    df["Base_MSRP"],
    bins=[-1, 25000, 50000, 75000, 100000, np.inf],
    labels=["Under 25K", "25K–50K", "50K–75K", "75K–100K", "100K+"]
)

df["CAFV_Flag"] = np.where(
    df["CAFV_Eligibility"].astype(str).str.contains(
        "eligible", case=False, na=False
    ),
    "Eligible",
    "Other"
)

df["Model_Year_Group"] = pd.cut(
    df["Model_Year"],
    bins=[-np.inf, 2015, 2019, 2022, 2024, np.inf],
    labels=["≤2015", "2016–2019", "2020–2022", "2023–2024", "2025+"]
)

display(df[[
    "Model_Year", "Vehicle_Age", "Age_Group",
    "Electric_Range", "Range_Category",
    "Base_MSRP", "MSRP_Category",
    "CAFV_Flag", "Model_Year_Group"
]].head())


,Model_Year,Vehicle_Age,Age_Group,Electric_Range,Range_Category,Base_MSRP,MSRP_Category,CAFV_Flag,Model_Year_Group
0,2020,4,3–5 Years,258,201–300,0,Under 25K,Eligible,2020–2022
1,2022,2,0–2 Years,25,0–100,0,Under 25K,Eligible,2020–2022
2,2023,1,0–2 Years,25,0–100,0,Under 25K,Eligible,2023–2024
3,2018,6,6–10 Years,215,201–300,0,Under 25K,Eligible,2016–2019
4,2018,6,6–10 Years,97,0–100,0,Under 25K,Eligible,2016–2019


## 4. Executive KPI Table

Core cards for the Power BI Overview page.


In [4]:
powerbi_kpis = pd.DataFrame({
    "KPI": [
        "Total Vehicles",
        "Unique Manufacturers",
        "Unique Models",
        "States Covered",
        "EV Types",
        "Average Electric Range",
        "Median Electric Range",
        "Average MSRP",
        "Median Vehicle Age"
    ],
    "Value": [
        len(df),
        df["Make"].nunique(),
        df["Model"].nunique(),
        df["State"].nunique(),
        df["EV_Type"].nunique(),
        df["Electric_Range"].mean(),
        df["Electric_Range"].median(),
        df["Base_MSRP"].mean(),
        df["Vehicle_Age"].median()
    ]
})

display(powerbi_kpis)


,KPI,Value
0,Total Vehicles,"150,482.00"
1,Unique Manufacturers,37.00
2,Unique Models,127.00
3,States Covered,41.00
4,EV Types,2.00
5,Average Electric Range,67.88
6,Median Electric Range,18.00
7,Average MSRP,"1,312.64"
8,Median Vehicle Age,3.00


## 5. Manufacturer Summary

Use this table for manufacturer ranking, market share, range, pricing and portfolio visuals.


In [5]:
manufacturer_summary = (
    df.groupby("Make", dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Model_Count=("Model", "nunique"),
          State_Count=("State", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Median_Range=("Electric_Range", "median"),
          Avg_MSRP=("Base_MSRP", "mean"),
          Median_MSRP=("Base_MSRP", "median"),
          Avg_Vehicle_Age=("Vehicle_Age", "mean")
      )
      .reset_index()
)

manufacturer_summary["Market_Share_%"] = (
    manufacturer_summary["Vehicle_Count"]
    / manufacturer_summary["Vehicle_Count"].sum() * 100
)

manufacturer_summary["Volume_Rank"] = (
    manufacturer_summary["Vehicle_Count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

display(manufacturer_summary.sort_values("Vehicle_Count", ascending=False).head(20))


,Make,Vehicle_Count,Model_Count,State_Count,Avg_Range,Median_Range,Avg_MSRP,Median_MSRP,Avg_Vehicle_Age,Market_Share_%,Volume_Rank
31,TESLA,68983,5,35,90.30,0.00,"1,648.60",0.00,3.36,45.84,1
25,NISSAN,13497,2,13,84.72,84.00,0.00,0.00,7.22,8.97,2
6,CHEVROLET,12026,5,14,95.27,53.00,0.00,0.00,5.81,7.99,3
10,FORD,7614,8,12,13.02,19.00,0.00,0.00,5.10,5.06,4
4,BMW,6439,11,11,38.16,30.00,"4,154.12",0.00,4.30,4.28,5
16,KIA,6198,7,9,52.62,26.00,"3,253.04",0.00,2.98,4.12,6
33,TOYOTA,5223,6,12,26.92,25.00,0.00,0.00,5.45,3.47,7
34,VOLKSWAGEN,4074,2,8,28.33,0.00,0.00,0.00,3.11,2.71,8
35,VOLVO,3536,7,7,15.63,18.00,"4,785.93",0.00,2.73,2.35,9
15,JEEP,3292,2,8,22.33,21.00,0.00,0.00,1.60,2.19,10


## 6. Model Summary


In [6]:
model_summary = (
    df.groupby(["Make", "Model"], dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          State_Count=("State", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Median_Range=("Electric_Range", "median"),
          Avg_MSRP=("Base_MSRP", "mean"),
          Median_MSRP=("Base_MSRP", "median"),
          Avg_Vehicle_Age=("Vehicle_Age", "mean")
      )
      .reset_index()
)

model_summary["Market_Share_%"] = (
    model_summary["Vehicle_Count"]
    / model_summary["Vehicle_Count"].sum() * 100
)

model_summary["Volume_Rank"] = (
    model_summary["Vehicle_Count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

display(model_summary.sort_values("Vehicle_Count", ascending=False).head(20))


,Make,Model,Vehicle_Count,State_Count,Avg_Range,Median_Range,Avg_MSRP,Median_MSRP,Avg_Vehicle_Age,Market_Share_%,Volume_Rank
108,TESLA,MODEL Y,28502,21,23.26,0.00,0.00,0.00,1.97,18.94,1
105,TESLA,MODEL 3,27709,29,121.38,215.00,0.00,0.00,3.59,18.41,2
91,NISSAN,LEAF,13187,13,86.72,84.00,0.00,0.00,7.37,8.76,3
106,TESLA,MODEL S,7611,12,183.46,208.00,"14,288.86",0.00,6.89,5.06,4
30,CHEVROLET,BOLT EV,5733,6,157.58,238.00,0.00,0.00,4.23,3.81,5
107,TESLA,MODEL X,5114,11,155.54,200.00,0.00,0.00,4.51,3.40,6
33,CHEVROLET,VOLT,4890,10,45.36,53.00,0.00,0.00,8.64,3.25,7
118,VOLKSWAGEN,ID.4,2999,8,0.00,0.00,0.00,0.00,1.89,1.99,8
62,KIA,NIRO,2876,6,80.40,26.00,0.00,0.00,3.13,1.91,9
34,CHRYSLER,PACIFICA,2642,11,32.24,32.00,"1,771.16",0.00,3.28,1.76,10


## 7. State / Geographic Summary


In [7]:
state_summary = (
    df.groupby("State", dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Median_Range=("Electric_Range", "median"),
          Avg_MSRP=("Base_MSRP", "mean"),
          Avg_Vehicle_Age=("Vehicle_Age", "mean")
      )
      .reset_index()
)

state_summary["Market_Share_%"] = (
    state_summary["Vehicle_Count"]
    / state_summary["Vehicle_Count"].sum() * 100
)

state_summary["State_Rank"] = (
    state_summary["Vehicle_Count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

display(state_summary.sort_values("Vehicle_Count", ascending=False).head(20))


,State,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_Range,Median_Range,Avg_MSRP,Avg_Vehicle_Age,Market_Share_%,State_Rank
39,WA,150141,37,127,67.86,18.00,"1,311.28",3.99,99.77,1
6,CA,92,16,28,69.20,22.00,"1,998.91",4.83,0.06,2
38,VA,35,10,18,73.11,32.00,0.00,5.14,0.02,3
21,MD,33,14,20,57.36,30.00,"3,090.91",4.33,0.02,4
36,TX,20,8,12,80.35,23.00,"1,749.75",4.80,0.01,5
26,NC,13,4,6,40.69,0.00,0.00,3.85,0.01,6
15,IL,12,6,7,94.25,30.50,"3,332.92",4.50,0.01,7
7,CO,11,6,10,90.73,32.00,0.00,4.00,0.01,8
4,AZ,11,8,10,36.82,22.00,0.00,4.18,0.01,8
11,FL,9,4,6,123.44,42.00,0.00,5.00,0.01,9


## 8. Yearly Trend Table


In [8]:
yearly_summary = (
    df.groupby("Model_Year", dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Median_Range=("Electric_Range", "median"),
          Avg_MSRP=("Base_MSRP", "mean"),
          Avg_Vehicle_Age=("Vehicle_Age", "mean")
      )
      .reset_index()
      .sort_values("Model_Year")
)

yearly_summary["YoY_Growth_%"] = (
    yearly_summary["Vehicle_Count"].pct_change() * 100
)

display(yearly_summary)


,Model_Year,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_Range,Median_Range,Avg_MSRP,Avg_Vehicle_Age,YoY_Growth_%
0,1997,1,1,1,39.00,39.00,0.00,27.00,NaN
1,1998,1,1,1,58.00,58.00,0.00,26.00,0.00
2,1999,4,1,1,74.00,74.00,0.00,25.00,300.00
3,2000,8,1,1,58.00,58.00,0.00,24.00,100.00
4,2002,2,1,1,95.00,95.00,0.00,22.00,-75.00
5,2003,1,1,1,95.00,95.00,0.00,21.00,-50.00
6,2008,19,1,1,220.00,220.00,"98,950.00",16.00,"1,800.00"
7,2010,24,2,2,226.88,245.00,"101,205.62",14.00,26.32
8,2011,796,5,5,70.85,73.00,958.54,13.00,"3,216.67"
9,2012,1633,8,9,62.19,35.00,"5,977.10",12.00,105.15


## 9. EV Type Summary


In [9]:
ev_type_summary = (
    df.groupby("EV_Type", dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Median_Range=("Electric_Range", "median"),
          Avg_MSRP=("Base_MSRP", "mean")
      )
      .reset_index()
)

ev_type_summary["Market_Share_%"] = (
    ev_type_summary["Vehicle_Count"]
    / ev_type_summary["Vehicle_Count"].sum() * 100
)

display(ev_type_summary)


,EV_Type,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_Range,Median_Range,Avg_MSRP,Market_Share_%
0,Battery Electric Vehicle (BEV),116807,27,67,78.61,0.00,"1,146.23",77.62
1,Plug-in Hybrid Electric Vehicle (PHEV),33675,25,63,30.66,26.00,"1,889.88",22.38


## 10. Manufacturer × EV Type

Power BI can use this table for stacked/clustered charts.


In [10]:
manufacturer_ev_type = (
    df.groupby(["Make", "EV_Type"], dropna=False)
      .size()
      .reset_index(name="Vehicle_Count")
)

manufacturer_ev_type["Market_Share_%"] = (
    manufacturer_ev_type["Vehicle_Count"]
    / manufacturer_ev_type["Vehicle_Count"].sum() * 100
)

display(
    manufacturer_ev_type
    .sort_values("Vehicle_Count", ascending=False)
    .head(30)
)


,Make,EV_Type,Vehicle_Count,Market_Share_%
44,TESLA,Battery Electric Vehicle (BEV),68983,45.84
36,NISSAN,Battery Electric Vehicle (BEV),13497,8.97
9,CHEVROLET,Battery Electric Vehicle (BEV),7136,4.74
47,TOYOTA,Plug-in Hybrid Electric Vehicle (PHEV),5034,3.35
10,CHEVROLET,Plug-in Hybrid Electric Vehicle (PHEV),4890,3.25
6,BMW,Plug-in Hybrid Electric Vehicle (PHEV),4643,3.09
22,KIA,Battery Electric Vehicle (BEV),4148,2.76
48,VOLKSWAGEN,Battery Electric Vehicle (BEV),4074,2.71
14,FORD,Battery Electric Vehicle (BEV),3908,2.60
15,FORD,Plug-in Hybrid Electric Vehicle (PHEV),3706,2.46


## 11. Range Analysis Table


In [11]:
range_summary = (
    df.groupby("Range_Category", observed=True)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_MSRP=("Base_MSRP", "mean"),
          Avg_Vehicle_Age=("Vehicle_Age", "mean")
      )
      .reset_index()
)

range_summary["Share_%"] = (
    range_summary["Vehicle_Count"]
    / range_summary["Vehicle_Count"].sum() * 100
)

display(range_summary)


,Range_Category,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_MSRP,Avg_Vehicle_Age,Share_%
0,0–100,113200,37,125,718.12,3.43,75.22
1,101–200,6613,9,9,379.90,5.94,4.39
2,201–300,28010,9,14,"4,060.17",5.81,18.61
3,301–400,2659,1,2,0.00,4.00,1.77


## 12. MSRP Analysis Table


In [12]:
msrp_summary = (
    df.groupby("MSRP_Category", observed=True)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Avg_Vehicle_Age=("Vehicle_Age", "mean")
      )
      .reset_index()
)

msrp_summary["Share_%"] = (
    msrp_summary["Vehicle_Count"]
    / msrp_summary["Vehicle_Count"].sum() * 100
)

display(msrp_summary)


,MSRP_Category,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_Range,Avg_Vehicle_Age,Share_%
0,Under 25K,147027,35,120,66.57,3.89,97.70
1,25K–50K,1154,6,7,59.89,6.70,0.77
2,50K–75K,2161,3,4,159.54,9.32,1.44
3,75K–100K,83,4,4,64.23,7.88,0.06
4,100K+,57,3,4,133.11,11.65,0.04


## 13. CAFV Analysis


In [13]:
cafv_summary = (
    df.groupby(["CAFV_Eligibility", "CAFV_Flag"], dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_Range=("Electric_Range", "mean")
      )
      .reset_index()
)

cafv_summary["Share_%"] = (
    cafv_summary["Vehicle_Count"]
    / cafv_summary["Vehicle_Count"].sum() * 100
)

display(cafv_summary)


,CAFV_Eligibility,CAFV_Flag,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_Range,Share_%
0,Clean Alternative Fuel Vehicle Eligible,Eligible,62951,29,57,156.65,41.83
1,Eligibility unknown as battery range has not b...,Other,69698,22,50,0.00,46.32
2,Not eligible due to low battery range,Eligible,17833,18,46,19.82,11.85


## 14. Dashboard Slicer Dimensions

Recommended slicers:

- Model Year
- State
- City
- County
- Make
- Model
- EV Type
- CAFV Eligibility
- Range Category
- MSRP Category
- Age Group

These dimension tables keep the Power BI report easy to filter.


In [14]:
dim_year = pd.DataFrame({
    "Model_Year": sorted(df["Model_Year"].dropna().unique())
})

dim_make = pd.DataFrame({
    "Make": sorted(df["Make"].dropna().unique())
})

dim_model = (
    df[["Make", "Model"]]
    .drop_duplicates()
    .sort_values(["Make", "Model"])
    .reset_index(drop=True)
)

dim_state = pd.DataFrame({
    "State": sorted(df["State"].dropna().unique())
})

dim_ev_type = pd.DataFrame({
    "EV_Type": sorted(df["EV_Type"].dropna().unique())
})

dim_cafv = pd.DataFrame({
    "CAFV_Eligibility": sorted(df["CAFV_Eligibility"].dropna().unique())
})

print("Dimension tables created.")


Dimension tables created.


## 15. Power BI Fact Table

This is the main table to import into Power BI.

Keep the granular vehicle-level records so slicers can interact with every analytical dimension.


In [15]:
fact_columns = [
    "VIN", "County", "City", "State", "Postal_Code",
    "Model_Year", "Make", "Model", "EV_Type",
    "CAFV_Eligibility", "CAFV_Flag",
    "Electric_Range", "Range_Category",
    "Base_MSRP", "MSRP_Category",
    "Legislative District", "Vehicle_ID",
    "Vehicle_Location", "Electric_Utility",
    "Census_Tract", "Vehicle_Age",
    "Age_Group", "Model_Year_Group"
]

fact_columns = [c for c in fact_columns if c in df.columns]

ev_powerbi_fact = df[fact_columns].copy()

display(ev_powerbi_fact.head())
print("Fact table shape:", ev_powerbi_fact.shape)


,VIN,County,City,State,Postal_Code,Model_Year,Make,Model,EV_Type,CAFV_Eligibility,CAFV_Flag,Electric_Range,Range_Category,Base_MSRP,MSRP_Category,Vehicle_ID,Vehicle_Location,Electric_Utility,Census_Tract,Vehicle_Age,Age_Group,Model_Year_Group
0,KM8K33AGXL,King,Seattle,WA,"98,103.00",2020,HYUNDAI,KONA,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,Eligible,258,201–300,0,Under 25K,249675142,POINT (-122.34301 47.659185),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),"53,033,004,800.00",4,3–5 Years,2020–2022
1,1C4RJYB61N,King,Bothell,WA,"98,011.00",2022,JEEP,GRAND CHEROKEE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,Eligible,25,0–100,0,Under 25K,233928502,POINT (-122.20578 47.762405),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),"53,033,021,804.00",2,0–2 Years,2020–2022
2,1C4RJYD61P,Yakima,Yakima,WA,"98,908.00",2023,JEEP,GRAND CHEROKEE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,Eligible,25,0–100,0,Under 25K,229675939,POINT (-120.6027202 46.5965625),PACIFICORP,"53,077,002,900.00",1,0–2 Years,2023–2024
3,5YJ3E1EA7J,King,Kirkland,WA,"98,034.00",2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,Eligible,215,201–300,0,Under 25K,104714466,POINT (-122.209285 47.71124),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),"53,033,021,903.00",6,6–10 Years,2016–2019
4,WBY7Z8C5XJ,Thurston,Olympia,WA,"98,501.00",2018,BMW,I3,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,Eligible,97,0–100,0,Under 25K,185498386,POINT (-122.89692 47.043535),PUGET SOUND ENERGY INC,"53,067,010,700.00",6,6–10 Years,2016–2019


Fact table shape: (150482, 22)


## 16. Export Power BI Data Layer

All outputs are CSV files so Power BI can load them directly.


In [16]:
exports = {
    "ev_powerbi_fact.csv": ev_powerbi_fact,
    "dim_year.csv": dim_year,
    "dim_make.csv": dim_make,
    "dim_model.csv": dim_model,
    "dim_state.csv": dim_state,
    "dim_ev_type.csv": dim_ev_type,
    "dim_cafv.csv": dim_cafv,
    "manufacturer_summary.csv": manufacturer_summary,
    "model_summary.csv": model_summary,
    "state_summary.csv": state_summary,
    "yearly_summary.csv": yearly_summary,
    "ev_type_summary.csv": ev_type_summary,
    "manufacturer_ev_type.csv": manufacturer_ev_type,
    "range_summary.csv": range_summary,
    "msrp_summary.csv": msrp_summary,
    "cafv_summary.csv": cafv_summary,
    "powerbi_kpis.csv": powerbi_kpis
}

for filename, table in exports.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)

print(f"✅ Exported {len(exports)} Power BI tables to:")
print(OUTPUT_DIR.resolve())


✅ Exported 17 Power BI tables to:
D:\ev_project_v2-Data Analysis\datas\powerbi


# 🎨 Power BI Dashboard Blueprint

## Page 1 — Executive Overview

### KPI cards

- Total Vehicles
- Manufacturers
- Models
- States
- Average Range
- Median Range

### Visuals

1. EV population by model year — **Line chart**
2. Top manufacturers — **Horizontal bar**
3. EV type share — **Donut**
4. Top states — **Horizontal bar**
5. Range category distribution — **Column chart**

### Slicers

`Model Year | State | Make | EV Type | CAFV Eligibility`

---

## Page 2 — Manufacturer Intelligence

### KPIs

- Selected manufacturer vehicles
- Model count
- State presence
- Median range

### Visuals

- Manufacturer ranking
- Make × EV Type
- Vehicle count vs median range
- Manufacturer × MSRP
- Manufacturer × age

### Slicers

`Make | EV Type | Model Year | State`

---

## Page 3 — Model Intelligence

### Visuals

- Top models
- Model range comparison
- Model MSRP comparison
- Model age distribution
- Make → Model drill-down

### Slicers

`Make | Model | EV Type | Range Category`

---

## Page 4 — Geographic Intelligence

### Visuals

- State vehicle population
- State market share
- Manufacturer diversity by state
- Model diversity by state
- State × EV Type

### Slicers

`State | County | City | Make | EV Type`

---

## Page 5 — EV Technology

### Visuals

- BEV vs PHEV
- EV type by year
- EV type by manufacturer
- Range distribution
- Range category by EV type

### Slicers

`EV Type | Model Year | Make | Range Category`

---

## Page 6 — Vehicle Economics

### Visuals

- MSRP distribution
- MSRP category share
- MSRP vs electric range
- Manufacturer median MSRP
- Model-year MSRP trend

### Slicers

`MSRP Category | Make | Model | EV Type`

---

## Page 7 — CAFV & Eligibility

### Visuals

- Eligibility distribution
- Eligibility by manufacturer
- Eligibility by state
- Eligibility by EV type
- Eligibility by model year

### Slicers

`CAFV Eligibility | Make | State | EV Type`

---

# 🎨 Theme Direction

Use a **cream/light professional theme**:

```text
Background       → Warm cream
Cards            → Off-white
Text             → Dark charcoal
Accent           → One consistent accent family
Gridlines        → Very subtle
Borders          → Minimal
```

Avoid excessive colors. Use consistent semantic colors for the same EV categories across every page.

---

# 🔗 Recommended Power BI Relationships

Use the vehicle-level fact table as the central analytical table.

```text
                    dim_year
                       │
dim_make ──────── ev_powerbi_fact ──────── dim_state
                       │
                  dim_ev_type
                       │
                   dim_cafv
```

`dim_model` can be related through the Make + Model key if you create a composite model key.

---

# ✅ Final Power BI Checklist

- [ ] Import `ev_powerbi_fact.csv`
- [ ] Import dimension tables
- [ ] Create relationships
- [ ] Create date/model-year hierarchy
- [ ] Create KPI measures
- [ ] Add slicers
- [ ] Add page navigation
- [ ] Add tooltips
- [ ] Add drill-through
- [ ] Apply cream theme
- [ ] Format numbers consistently
- [ ] Add source/data-quality page
- [ ] Test every slicer
- [ ] Test cross-filtering
- [ ] Validate totals against Python
- [ ] Export `.pbix`

## Important

**Python prepares the analytical data layer. Power BI handles the final interactive dashboard.**

This separation keeps the project professional and reproducible.
